In [ ]:
# Run relative to the repository root so data/, models/ and figures/ resolve.
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

In [5]:
import pandas as pd
import numpy as np
from pycaret.classification import load_model, predict_model
from sklearn.calibration import CalibratedClassifierCV
from catboost import CatBoostClassifier

blend = load_model("models/permdi_blend_ridge_cat_v1")  # ✓

# 1) Inspect the pipeline structure
print("Pipeline steps:", [name for name, _ in blend.steps])

# 2) Grab the trained VotingClassifier from the pipeline
#    (PyCaret names the last step 'trained_model' in v3.x)
try:
    voter = blend.named_steps["trained_model"]
except KeyError:
    # Fallbacks if the name differs in your env
    voter = getattr(blend, "_final_estimator", None) or blend.steps[-1][1]

print("Final estimator type:", type(voter))

# 3) List base estimators inside the VotingClassifier
#    (scikit-learn exposes .estimators_ after fit, or .estimators before fit)
base_ests = getattr(voter, "estimators_", None) or getattr(voter, "estimators", None)
print("Num base estimators:", len(base_ests))

# 4) Sanity check types (should be Calibrated Ridge and CatBoost)
print([type(e) for e in base_ests])

# Optional asserts to prove what’s inside:
assert any(isinstance(e, CalibratedClassifierCV) for e in base_ests), "Calibrated Ridge missing?"
assert any(isinstance(e, CatBoostClassifier) for e in base_ests), "CatBoost missing?"


Transformation Pipeline and Model Successfully Loaded
Pipeline steps: ['numerical_imputer', 'categorical_imputer', 'ordinal_encoding', 'onehot_encoding', 'balance', 'clean_column_names', 'actual_estimator']
Final estimator type: <class 'sklearn.ensemble._voting.VotingClassifier'>
Num base estimators: 2
[<class 'sklearn.calibration.CalibratedClassifierCV'>, <class 'catboost.core.CatBoostClassifier'>]


In [8]:
import pandas as pd
import numpy as np
from pycaret.classification import load_model, predict_model
from sklearn.calibration import CalibratedClassifierCV
from catboost import CatBoostClassifier

recurrence = load_model("models/recurrence_nb_v1")  # ✓

# 1) Inspect the pipeline structure
print("Pipeline steps:", [name for name, _ in blend.steps])

# 2) Grab the trained VotingClassifier from the pipeline
#    (PyCaret names the last step 'trained_model' in v3.x)
try:
    voter = blend.named_steps["trained_model"]
except KeyError:
    # Fallbacks if the name differs in your env
    voter = getattr(blend, "_final_estimator", None) or blend.steps[-1][1]

print("Final estimator type:", type(voter))

# 3) List base estimators inside the VotingClassifier
#    (scikit-learn exposes .estimators_ after fit, or .estimators before fit)
base_ests = getattr(voter, "estimators_", None) or getattr(voter, "estimators", None)
print("Num base estimators:", len(base_ests))

# 4) Sanity check types (should be Calibrated Ridge and CatBoost)
print([type(e) for e in base_ests])

# Optional asserts to prove what’s inside:
assert any(isinstance(e, CalibratedClassifierCV) for e in base_ests), "Calibrated Ridge missing?"
assert any(isinstance(e, CatBoostClassifier) for e in base_ests), "CatBoost missing?"


Transformation Pipeline and Model Successfully Loaded
Pipeline steps: ['numerical_imputer', 'categorical_imputer', 'ordinal_encoding', 'onehot_encoding', 'balance', 'clean_column_names', 'actual_estimator']
Final estimator type: <class 'sklearn.ensemble._voting.VotingClassifier'>
Num base estimators: 2
[<class 'sklearn.calibration.CalibratedClassifierCV'>, <class 'catboost.core.CatBoostClassifier'>]


In [10]:
# Minimal smoke test on a few rows from your original CSV
FEATURES_DI = [
    "Age ", "BMI", "Tumor size", "Tumor size.1", "Tumor size.2", "Tumor volume",
    "Last BMI on file",
    "Pre-op visual impairment (y/n) - symptomatic complaint of visual loss",
    "Pre-op hydrocephalus (y/n)", "Pre-op intacranial pressure symptoms",
    "Pre-op headache", "Pre-op endocrine changes (y/n)",
    "Pre-op panhypopit (y/n) - define number of hormone axes that are non-functional",
    "Intraventricular extension",
    "Surgical Approach", "Resection outcome (total, subtotal, partial)"
]

FEATURES_RECURRENCE = FEATURES_DI + [
    "prior radiation", "BRAF+"
]

def prepare_inputs(df, features: list[str]) -> pd.DataFrame:
    df = df.copy()
    # recompute tumor volume if needed
    if "Tumor volume" not in df.columns and all(c in df.columns for c in ["Tumor size","Tumor size.1","Tumor size.2"]):
        df["Tumor volume"] = (4/3)*np.pi * (df["Tumor size"]/2)*(df["Tumor size.1"]/2)*(df["Tumor size.2"]/2)
    # ensure all expected columns exist
    for c in features:
        if c not in df.columns:
            df[c] = np.nan
    # light coercion for binary-like cols
    bin_like = [
        "Pre-op visual impairment (y/n) - symptomatic complaint of visual loss",
        "Pre-op hydrocephalus (y/n)", "Pre-op intacranial pressure symptoms",
        "Pre-op headache", "Pre-op endocrine changes (y/n)",
        "Pre-op panhypopit (y/n) - define number of hormone axes that are non-functional",
        "Intraventricular extension",
    ]
    for c in bin_like:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    # keep exact column order
    return df[features]

df_raw = pd.read_csv("data/data.csv")
X_new_pred1 = prepare_inputs(df_raw, FEATURES_DI).head(5)
pred1 = predict_model(blend, data=X_new_pred1)
X_new_pred2 = prepare_inputs(df_raw, FEATURES_RECURRENCE).head(5)
pred2 = predict_model(recurrence, data=X_new_pred2)
print(pred1[["prediction_label","prediction_score"]])
print(pred2[["prediction_label","prediction_score"]])


   prediction_label  prediction_score
0                 0            0.6172
1                 0            0.7061
2                 1            0.5628
3                 1            0.6767
4                 1            0.5477
   prediction_label  prediction_score
0                 0            0.9600
1                 0            0.7813
2                 1            0.6933
3                 0            0.6595
4                 0            0.6731


In [38]:
import requests, json

base = "https://precision-craniopharyngiomas.onrender.com"

payload = {
    "Age ": 60, "BMI": 28.3,
    "Tumor size": 3.4, "Tumor size.1": 2.2, "Tumor size.2": 1.9,
    "Last BMI on file": 29.5,
    "Pre-op visual impairment (y/n) - symptomatic complaint of visual loss": 1,
    "Pre-op hydrocephalus (y/n)": 0, "Pre-op intacranial pressure symptoms": 0,
    "Pre-op headache": 1, "Pre-op endocrine changes (y/n)": 1,
    "Pre-op panhypopit (y/n) - define number of hormone axes that are non-functional": 0,
    "Intraventricular extension": 0,
    "Surgical Approach": "craniotomy",
    "Resection outcome (total, subtotal, partial)": "subtotal"
}

def call(endpoint):
    url = f"{base}/{endpoint}"
    try:
        r = requests.post(url, json=payload, timeout=20)
        print(f"\n[{endpoint.upper()}] Status:", r.status_code)
        if r.status_code != 200:
            print("Text:", r.text)
        else:
            print("JSON:", json.dumps(r.json(), indent=2))
    except Exception as e:
        print(f"Error contacting {endpoint}:", e)

# call both endpoints
call("predict")
call("predict-recurrence")



[PREDICT] Status: 200
JSON: {
  "prob_permanent_DI": 0.6258092026670323,
  "label": 1,
  "threshold": 0.5
}

[PREDICT-RECURRENCE] Status: 200
JSON: {
  "prob_recurrence": 0.8621825853994396,
  "label": 1,
  "threshold": 0.5
}


In [39]:
# --- Imports
import math
import numpy as np
import pandas as pd
from pycaret.classification import load_model, predict_model

# --- Feature schema
FEATURES_DI = [
    "Age ", "BMI", "Tumor size", "Tumor size.1", "Tumor size.2", "Tumor volume",
    "Last BMI on file",
    "Pre-op visual impairment (y/n) - symptomatic complaint of visual loss",
    "Pre-op hydrocephalus (y/n)", "Pre-op intacranial pressure symptoms",
    "Pre-op headache", "Pre-op endocrine changes (y/n)",
    "Pre-op panhypopit (y/n) - define number of hormone axes that are non-functional",
    "Intraventricular extension",
    "Surgical Approach", "Resection outcome (total, subtotal, partial)",
]

FEATURES_RECURRENCE = FEATURES_DI + ["prior radiation", "BRAF+"]

BIN_LIKE = [
    "Pre-op visual impairment (y/n) - symptomatic complaint of visual loss",
    "Pre-op hydrocephalus (y/n)", "Pre-op intacranial pressure symptoms",
    "Pre-op headache", "Pre-op endocrine changes (y/n)",
    "Pre-op panhypopit (y/n) - define number of hormone axes that are non-functional",
    "Intraventricular extension",
    "prior radiation", "BRAF+",
]

def prepare_inputs(df: pd.DataFrame, features: list[str]) -> pd.DataFrame:
    """Mirror backend prepare_inputs for recurrence."""
    df = df.copy()

    # Recompute Tumor volume if missing from axes (ellipsoid)
    if "Tumor volume" not in df.columns and all(c in df.columns for c in ["Tumor size","Tumor size.1","Tumor size.2"]):
        df["Tumor volume"] = (4/3)*math.pi * (df["Tumor size"]/2)*(df["Tumor size.1"]/2)*(df["Tumor size.2"]/2)

    # Ensure columns exist
    for c in features:
        if c not in df.columns:
            df[c] = np.nan

    # Coerce binary-like to numeric
    for c in BIN_LIKE:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    # Normalize categories
    if "Surgical Approach" in df.columns:
        df["Surgical Approach"] = (
            df["Surgical Approach"]
              .astype(str).str.strip().str.lower()
              .replace({"endo": "endoscopic", "endo.": "endoscopic"})
        )

    if "Resection outcome (total, subtotal, partial)" in df.columns:
        col = "Resection outcome (total, subtotal, partial)"
        df[col] = (
            df[col].astype(str).str.strip().str.lower()
                  .replace({"near-total": "near total", "near_total": "near total"})
        )

    # Keep order
    return df[features]

# Load trained isotonic-calibrated NB pipeline
recurrence = load_model("models/recurrence_blend_nb_et_v1")  # file 'models/recurrence_nb_v1.pkl'

# Load a small slice of your dataset for smoke tests
df_raw = pd.read_csv("data/data.csv")
assert isinstance(df_raw, pd.DataFrame) and len(df_raw) > 0, "data.csv not found or empty."
df_raw.head(3)

def run_predict_rows(df_rows: pd.DataFrame):
    X = prepare_inputs(df_rows, FEATURES_RECURRENCE)
    out = predict_model(recurrence, data=X)
    return X, out

# 1) Predict on first 5 rows with full recurrence schema
X1, out1 = run_predict_rows(df_raw.head(5))
assert "prediction_label" in out1.columns and "prediction_score" in out1.columns
assert out1["prediction_score"].between(0, 1).all(), "Scores must be in [0,1]"
print("Smoke test ✓", out1[["prediction_label","prediction_score"]].head())

X_det = prepare_inputs(df_raw.head(5), FEATURES_RECURRENCE)
oA = predict_model(recurrence, data=X_det)["prediction_score"].round(10).tolist()
oB = predict_model(recurrence, data=X_det)["prediction_score"].round(10).tolist()
assert oA == oB, "Non-deterministic outputs for identical inputs."
print("Determinism ✓")


Transformation Pipeline and Model Successfully Loaded
Smoke test ✓    prediction_label  prediction_score
0                 1            0.5092
1                 0            0.8873
2                 0            0.6374
3                 0            0.8250
4                 0            0.8352
Determinism ✓


In [40]:
row = df_raw.sample(1, random_state=0).copy()

# Force NaNs in optional fields
row_missing = row.copy()
row_missing["BRAF+"] = np.nan
row_missing["prior radiation"] = np.nan

# Weird casing/aliases for categories
row_weird = row.copy()
row_weird["Surgical Approach"] = "Endo."
row_weird["Resection outcome (total, subtotal, partial)"] = "Near-Total"

# Should run without raising
_, out_missing = run_predict_rows(row_missing)
_, out_weird   = run_predict_rows(row_weird)

assert out_missing["prediction_score"].between(0,1).all()
assert out_weird["prediction_score"].between(0,1).all()
print("Missing & normalization ✓")

row = df_raw.sample(1, random_state=1).copy()
# Add extra unrelated column
row["IGNORED_EXTRA"] = 12345

# Shuffle columns
row_shuffled = row.sample(frac=1, axis=1, random_state=2)

# Both should give valid predictions
_, out_orig     = run_predict_rows(row)
_, out_shuffled = run_predict_rows(row_shuffled)

assert out_orig["prediction_score"].between(0,1).all()
assert out_shuffled["prediction_score"].between(0,1).all()
print("Column order/extraneous ✓")

row = df_raw.sample(8, random_state=3)
X, out = run_predict_rows(row)

# Recompute labels at 0.5 and compare with PyCaret's label column if it's 0/1
# (PyCaret's "prediction_label" may be bool/str; coerce to 0/1 if possible)
scores = out["prediction_score"].astype(float).values
my_labels = (scores >= 0.5).astype(int)

try:
    pc_labels = pd.to_numeric(out["prediction_label"], errors="coerce").fillna(-1).astype(int).values
    # Only compare where PyCaret yields 0/1
    mask = pc_labels >= 0
    if mask.any():
        assert (my_labels[mask] == pc_labels[mask]).all(), "Label mismatch vs threshold=0.5"
    print("Label/threshold ✓")
except Exception:
    print("Label column not strictly numeric; skipped strict comparison ✓")


Missing & normalization ✓
Column order/extraneous ✓
Label column not strictly numeric; skipped strict comparison ✓


In [51]:
# Minimal payload: required numerics + categories; rest will be NaN
case_min = pd.DataFrame([{
    "Age ": 59,
    "BMI": 25,
    "Tumor size":2.0,
    "Tumor size.1": 5.0,
    "Tumor size.2": 4.0,
    "Surgical Approach": "endoscopic",
    "Resection outcome (total, subtotal, partial)": "total",
    # Recurrence-only fields provided as yes/no numerics (frontend constraint):
    "BRAF+": 1,
    "prior radiation": 0,
}])

X, out = run_predict_rows(case_min)
assert out["prediction_score"].between(0,1).all()
print("Minimal payload ✓", out[["prediction_label","prediction_score"]])


Minimal payload ✓    prediction_label  prediction_score
0                 0            0.6155
